In [ ]:
import os
import re
from collections import Counter
import openpyxl
from openpyxl.styles import PatternFill


# ======================================================================
# PART 1: COST AUDIT + COLOUR HIGHLIGHTING
# ======================================================================
LAST_RUN = {}   # counts from the most recent run (used by the interface)


def read_rows(path, sheet_name):
    """Return (sheet_title, list_of_row_lists) from an .xlsb or .xlsx file."""
    if os.path.splitext(path)[1].lower() == ".xlsb":
        from pyxlsb import open_workbook as open_xlsb

        with open_xlsb(path) as wb:
            name = sheet_name if sheet_name in wb.sheets else wb.sheets[0]
            with wb.get_sheet(name) as sh:
                return name, [[c.v for c in row] for row in sh.rows()]

    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb.worksheets[0]
    name, rows = ws.title, [list(r) for r in ws.iter_rows(values_only=True)]
    wb.close()
    return name, rows


def parse_rate(val):
    """Strictly evaluates numerical rate values from cells."""
    if val is None:
        return 0.0
    if isinstance(val, (int, float)):
        return float(val) if val > 0 else 0.0

    val_str = str(val).replace("\xa0", " ").strip().upper()
    ignored_tokens = {
        "POP",
        "INHOUSE",
        "IN-HOUSE",
        "IN HOUSE",
        "0",
        "0.0",
        "0.00",
        "-",
        "--",
        "",
        "NONE",
        "NAN",
        "NIL",
        "NA",
        "N/A",
        "F/R",
    }
    if val_str in ignored_tokens:
        return 0.0

    clean_str = re.sub(r"[^\d.]", "", val_str)
    if not clean_str or clean_str == ".":
        return 0.0

    try:
        num = float(clean_str)
        return num if num > 0.0 else 0.0
    except (ValueError, TypeError):
        return 0.0


def parse_level(val):
    """Extracts integer level from cell value."""
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return int(val)
    m = re.search(r"\d+", str(val).strip())
    return int(m.group()) if m else None


def generate_comprehensive_bom_audit(
    new_input_file="Qute EV MPL Sep26.xlsx",
    old_input_file="qute mpl old data.xlsx",
    output_file="Qute_EV_MPL_Comprehensive_Audited.xlsx",
    sheet_name="1. R140B Qute EV",
    header_row=5,
    level_col_idx=2,  # Column C (Level) -> 0-based index 2
    category_col_idx=8,  # Column I (Category) -> 0-based index 8
    rate_p_col_idx=15,  # Column P -> 0-based index 15
    rate_u_col_idx=20,  # Column U in the new sheet -> 0-based index 20
    old_col_u_idx=21,  # Column U in old file for green check (1-based)
):
    print("=" * 80)
    print("Starting Comprehensive BOM Audit & Independent Highlighting Pipeline:")
    print(" 1. Level-wise Cost Duplication (Yellow)")
    print(" 2. Column P vs Old Data Column U Mismatch (Green)")
    print(
        " 3. Full MPL Scan: Absolute % Change (Col P vs Col U) with Category / 6% Fallback (Blue)"
    )
    print("=" * 80)

    # Category percentage thresholds dictionary
    category_thresholds = {
        "casting": 2.0,
        "elastomer": 3.0,
        "electrical": 4.0,
        "inhouse": 2.0,
        "f&m": 5.0,
        "prop": 6.0,
        "polymer": 4.5,
        "p&f": 5.0,
    }
    default_fallback_threshold = (
        6.0  # Fallback rule if category is unknown or missing
    )

    # ----------------------------------------------------
    # STEP 1: Ingest New File (.xlsb or .xlsx)
    # ----------------------------------------------------
    target_sheet, all_new_data = read_rows(new_input_file, sheet_name)
    print(f"Reading New Sheet: '{target_sheet}' from {new_input_file}")

    total_rows = len(all_new_data)
    print(f"Total Rows Ingested from New File: {total_rows:,}")

    # Parse BOM items for stack traversal (Yellow logic)
    bom_items = []
    for r_idx in range(header_row, total_rows):
        row_vals = all_new_data[r_idx]
        lvl_val = (
            row_vals[level_col_idx] if len(row_vals) > level_col_idx else None
        )
        cost_val = (
            row_vals[rate_p_col_idx] if len(row_vals) > rate_p_col_idx else None
        )

        lvl = parse_level(lvl_val)
        cost_num = parse_rate(cost_val)
        item_id_val = row_vals[4] if len(row_vals) > 4 else ""

        bom_items.append({
            "array_idx": r_idx,
            "excel_row": r_idx + 1,
            "level": lvl,
            "cost": cost_num,
            "has_cost": cost_num > 0.0,
            "item_id": str(item_id_val).strip() if item_id_val else "",
        })

    # Stack Traversal: Identify duplicate lower-level child costs (Yellow)
    active_ancestors = {}
    yellow_indices = set()

    for item in bom_items:
        curr_lvl = item["level"]
        curr_row = item["excel_row"]

        if curr_lvl is None:
            continue

        levels_to_remove = [lvl for lvl in active_ancestors if lvl >= curr_lvl]
        for lvl in levels_to_remove:
            del active_ancestors[lvl]

        costed_parent = None
        if active_ancestors:
            for p_lvl in sorted(active_ancestors.keys(), reverse=True):
                if active_ancestors[p_lvl]["has_cost"]:
                    costed_parent = active_ancestors[p_lvl]
                    break

        if costed_parent is not None and item["has_cost"]:
            yellow_indices.add(item["array_idx"])

        active_ancestors[curr_lvl] = {
            "level": curr_lvl,
            "item_id": item["item_id"],
            "cost": item["cost"],
            "has_cost": item["has_cost"],
            "excel_row": curr_row,
        }

    print(f"✓ Level-wise Duplicate Rows Identified (Yellow): {len(yellow_indices):,}")

    # ----------------------------------------------------
    # STEP 2: Ingest Old File & Process Green / Full Blue Scan
    # ----------------------------------------------------
    print(f"Reading Old File: {old_input_file} for Green Mismatch check...")
    wb_old = openpyxl.load_workbook(old_input_file, data_only=True)
    ws_old = wb_old.active

    green_indices = set()
    blue_indices = set()
    row_remarks = {}

    max_compare_rows = max(total_rows, ws_old.max_row)

    # Scan through ALL rows starting from the header row
    for r_idx in range(header_row, max_compare_rows):
        row_vals = list(all_new_data[r_idx]) if r_idx < total_rows else []

        # 1. Green Check (Old File Col U vs New File Col P)
        val_old_green = ws_old.cell(row=r_idx + 1, column=old_col_u_idx).value
        val_new_p = (
            row_vals[rate_p_col_idx]
            if len(row_vals) > rate_p_col_idx
            else None
        )
        if parse_rate(val_old_green) != parse_rate(val_new_p):
            green_indices.add(r_idx)

        # 2. Blue Check (Full MPL Scan: Absolute % Change between Column P and Column U)
        if r_idx < total_rows:
            category_val = ""
            if len(row_vals) > category_col_idx and row_vals[category_col_idx]:
                category_val = (
                    str(row_vals[category_col_idx])
                    .replace("\xa0", " ")
                    .strip()
                    .lower()
                )

            threshold = category_thresholds.get(
                category_val, default_fallback_threshold
            )

            val_u_new = (
                row_vals[rate_u_col_idx]
                if len(row_vals) > rate_u_col_idx
                else None
            )
            cost_p = parse_rate(val_new_p)
            cost_u = parse_rate(val_u_new)

            if cost_u > 0:
                # Use ABS() so both increases AND decreases beyond the threshold are captured
                pct_change = abs((cost_p - cost_u) / cost_u) * 100

                if pct_change > threshold:
                    blue_indices.add(r_idx)

                    # Determine status remark
                    row_str = " ".join([str(c).lower() for c in row_vals])
                    is_mod = (
                        "mod" in row_str
                        or "modified" in row_str
                        or "change" in row_str
                    )
                    if is_mod:
                        row_remarks[r_idx] = "It is a modified part"
                    else:
                        row_remarks[r_idx] = "Already existing part"

    wb_old.close()
    print(f"✓ Value Mismatch Rows Identified (Green): {len(green_indices):,}")
    print(
        f"✓ Full MPL Category/Fallback Threshold Breach Rows Identified (Blue): {len(blue_indices):,}"
    )

    LAST_RUN.update(rows=max(total_rows - header_row, 0), yellow=len(yellow_indices),
                    green=len(green_indices), blue=len(blue_indices))

    # ----------------------------------------------------
    # STEP 3: Build Output Workbook with Independent Highlighting
    # ----------------------------------------------------
    print(f"\nWriting audited output to: {output_file} ...")
    out_wb = openpyxl.Workbook()
    out_ws = out_wb.active
    out_ws.title = "Audited_BOM_Matrix"

    yellow_fill = PatternFill(
        start_color="FFFF99", end_color="FFFF99", fill_type="solid"
    )
    blue_fill = PatternFill(
        start_color="BDD7EE", end_color="BDD7EE", fill_type="solid"
    )
    green_fill = PatternFill(
        start_color="C6EFCE", end_color="C6EFCE", fill_type="solid"
    )

    if all_new_data:
        headers = list(all_new_data[header_row - 1])
        if "Audit_Remarks" not in headers:
            headers.append("Audit_Remarks")
        all_new_data[header_row - 1] = headers

    for r_idx, row_vals in enumerate(all_new_data):
        excel_row = r_idx + 1
        row_list = list(row_vals)

        if r_idx >= header_row:
            remark = row_remarks.get(r_idx, "")
            row_list.append(remark)

        out_ws.append(row_list)
        num_cols = len(row_list)

        # Independent Highlighting Application
        if r_idx in yellow_indices:
            for col_idx in range(1, num_cols + 1):
                out_ws.cell(row=excel_row, column=col_idx).fill = yellow_fill

        if r_idx in blue_indices:
            for col_idx in range(1, num_cols + 1):
                out_ws.cell(row=excel_row, column=col_idx).fill = blue_fill

        if r_idx in green_indices:
            for col_idx in range(1, num_cols + 1):
                out_ws.cell(row=excel_row, column=col_idx).fill = green_fill

    out_wb.save(output_file)
    print(f"✓ Success! Success file saved to: '{output_file}'")


# ======================================================================
# PART 2: SUB-GROUP MAPPING  (runs on the highlighted file from Part 1)
# Only writes values into the Sub Groups column, so existing fills are kept.
# ======================================================================
def map_subgroups(
    new_file="Qute_EV_MPL_Comprehensive_Audited.xlsx",
    old_file="qute mpl old data.xlsx",
    output_file="Qute_EV_MPL_Final.xlsx",
    header_row=5,
    level_col=3,       # Column C (Level)
    item_id_col=4,     # Column D (Item ID)
    subgroup_col=24,   # Column X (Sub Groups)
    assy_value="ASSY"  # Level 2 rows whose sub group is exactly this hand off to Level 3
):
    def resolve(path):
        """Try the given name, then space/underscore variants, so either spelling works."""
        for cand in (path, path.replace("_", " "), path.replace(" ", "_")):
            if os.path.exists(cand):
                return cand
        return None

    new_found, old_found = resolve(new_file), resolve(old_file)
    if not new_found or not old_found:
        xlsx = [f for f in os.listdir(".") if f.lower().endswith(".xlsx")]
        raise FileNotFoundError(
            f"Could not find: {[n for n, f in ((new_file, new_found), (old_file, old_found)) if not f]}\n"
            f"Looking in: {os.getcwd()}\n"
            f"Excel files found there: {xlsx}\n"
            "Fix the file names (or use full paths) in the call at the bottom of the script."
        )
    new_file, old_file = new_found, old_found

    def clean_id(val):
        if val is None:
            return ""
        s = str(val).strip()
        if s.endswith(".0"):
            s = s[:-2]
        return s.upper()

    def get_level(val):
        if val is None:
            return None
        m = re.search(r"\d+", str(val))
        return int(m.group()) if m else None

    def is_assy(sub):
        return sub is not None and str(sub).strip().upper() == assy_value.upper()

    # ---------- Item ID -> Sub Group from old file (first occurrence wins) ----------
    wb_old = openpyxl.load_workbook(old_file, data_only=True)
    ws_old = wb_old.active
    old_map = {}
    for r in range(header_row + 1, ws_old.max_row + 1):
        cid = clean_id(ws_old.cell(row=r, column=item_id_col).value)
        sub = ws_old.cell(row=r, column=subgroup_col).value
        sub = str(sub).strip() if sub is not None else ""
        if cid and sub and cid not in old_map:
            old_map[cid] = sub
    wb_old.close()
    print(f"Loaded {len(old_map)} item->subgroup mappings from old file.")

    wb_new = openpyxl.load_workbook(new_file)
    ws_new = wb_new.active

    # ---------- Count how many times each Item ID appears in the new sheet (all levels) ----------
    id_counts = Counter()
    for r in range(header_row + 1, ws_new.max_row + 1):
        if get_level(ws_new.cell(row=r, column=level_col).value) is None:
            continue
        cid = clean_id(ws_new.cell(row=r, column=item_id_col).value)
        if cid:
            id_counts[cid] += 1

    def next_l3_sub(r):
        """Succeeding Level 3 in the same Level 2 block: nearest one that is NOT itself repeated
        and has an old-data match (repeated ones get their value from neighbours, so skip them)."""
        for r2 in range(r + 1, ws_new.max_row + 1):
            l2 = get_level(ws_new.cell(row=r2, column=level_col).value)
            if l2 is None:
                continue
            if l2 <= 2:
                break
            if l2 == 3:
                c2 = clean_id(ws_new.cell(row=r2, column=item_id_col).value)
                if c2 in old_map and id_counts[c2] <= 1:
                    return old_map[c2]
        return None

    stats = dict(direct=0, repeat_from_above=0, repeat_fallback_old=0,
                 l3plus_inherit=0, assy_l2=0, assy_l3_own=0, assy_l3_blank=0,
                 assy_l4plus=0, assy_l4_own=0, assy_l4_blank=0,
                 l3rep_prev=0, l3rep_next=0, l3rep_l2=0, l3rep_own=0)

    current_l2 = None      # final sub group of the active Level 2 row
    current_l3 = None      # final sub group of the active Level 3 row (Assy mode only)
    last_l3 = None         # last non-empty Level 3 sub group in the current Level 2 block
    assy_mode = False      # True while under a Level 2 row whose sub group is exactly "Assy"
    prev_sub = None        # final sub group of the row immediately above

    # ---------- Single top-down pass ----------
    for r in range(header_row + 1, ws_new.max_row + 1):
        lvl = get_level(ws_new.cell(row=r, column=level_col).value)
        if lvl is None:
            continue
        cid = clean_id(ws_new.cell(row=r, column=item_id_col).value)
        final_sub = None

        if lvl <= 1:
            current_l2 = None
            current_l3 = None
            last_l3 = None
            assy_mode = False
            if cid in old_map:
                final_sub = old_map[cid]
                ws_new.cell(row=r, column=subgroup_col, value=final_sub)
                stats["direct"] += 1
            else:
                existing = ws_new.cell(row=r, column=subgroup_col).value
                final_sub = str(existing).strip() if existing not in (None, "") else None

        elif lvl == 2:
            current_l3 = None
            last_l3 = None
            if cid and id_counts[cid] > 1:
                # Repeated part at Level 2: copy from the row just above
                if prev_sub and not is_assy(prev_sub):
                    final_sub = prev_sub
                    stats["repeat_from_above"] += 1
                elif cid in old_map:
                    # nothing usable above ("Assy" is a placeholder, not a real sub group): use old data
                    final_sub = old_map[cid]
                    stats["repeat_fallback_old"] += 1
                elif prev_sub:
                    final_sub = prev_sub
                    stats["repeat_from_above"] += 1
            elif cid in old_map:
                final_sub = old_map[cid]
                stats["direct"] += 1
            ws_new.cell(row=r, column=subgroup_col, value=final_sub)
            current_l2 = final_sub
            assy_mode = is_assy(final_sub)       # sub group is only "Assy" -> hand off to Level 3
            if assy_mode:
                stats["assy_l2"] += 1

        elif assy_mode:
            # Level 2 is only "Assy": Level 3 is the reference for Level 4+
            if lvl == 3:
                if cid and id_counts[cid] > 1:
                    # Repeated part at Level 3: preceding L3 -> succeeding L3 -> preceding L2
                    if last_l3:
                        final_sub = last_l3
                        stats["l3rep_prev"] += 1
                    else:
                        nxt = next_l3_sub(r)
                        if nxt:
                            final_sub = nxt
                            stats["l3rep_next"] += 1
                        elif current_l2 and not is_assy(current_l2):
                            final_sub = current_l2
                            stats["l3rep_l2"] += 1
                        elif cid in old_map:
                            final_sub = old_map[cid]   # L2 is only the "Assy" placeholder: use old data
                            stats["l3rep_own"] += 1
                elif cid in old_map:
                    final_sub = old_map[cid]
                    stats["assy_l3_own"] += 1
                else:
                    final_sub = None
                    stats["assy_l3_blank"] += 1
                current_l3 = final_sub
                if final_sub:
                    last_l3 = final_sub
            else:  # Level 4 and below
                if current_l3:
                    final_sub = current_l3
                    stats["assy_l4plus"] += 1
                elif cid in old_map:
                    final_sub = old_map[cid]
                    stats["assy_l4_own"] += 1
                else:
                    stats["assy_l4_blank"] += 1
            ws_new.cell(row=r, column=subgroup_col, value=final_sub)

        else:
            # Normal Level 2: all Level 3+ take the Level 2 sub group
            if current_l2:
                final_sub = current_l2
                stats["l3plus_inherit"] += 1
            elif cid in old_map:
                final_sub = old_map[cid]
                stats["direct"] += 1
            ws_new.cell(row=r, column=subgroup_col, value=final_sub)

        prev_sub = final_sub

    lvl_rows = [r for r in range(header_row + 1, ws_new.max_row + 1)
                if get_level(ws_new.cell(row=r, column=level_col).value) is not None]
    LAST_RUN.update(sub_total=len(lvl_rows),
                    sub_filled=sum(1 for r in lvl_rows if ws_new.cell(row=r, column=subgroup_col).value))
    wb_new.save(output_file)
    wb_new.close()

    print(f"Direct old-data matches:                          {stats['direct']:,}")
    print(f"Repeated L2 parts copied from row above:          {stats['repeat_from_above']:,}")
    print(f"Repeated L2 parts, nothing usable above (used old):    {stats['repeat_fallback_old']:,}")
    print(f"Level 3+ under normal L2 given L2 sub group:      {stats['l3plus_inherit']:,}")
    print(f"Level 2 rows with sub group only '{assy_value.title()}':        {stats['assy_l2']:,}")
    print(f"  L3 under them using own old-data match:         {stats['assy_l3_own']:,}")
    print(f"  L3 under them with no match (left blank):       {stats['assy_l3_blank']:,}")
    print(f"  Repeated L3 copied from preceding L3:           {stats['l3rep_prev']:,}")
    print(f"  Repeated L3 copied from succeeding L3:          {stats['l3rep_next']:,}")
    print(f"  Repeated L3 copied from preceding L2:           {stats['l3rep_l2']:,}")
    print(f"  Repeated L3 fell back to own old-data match:    {stats['l3rep_own']:,}")
    print(f"  L4+ copied from their L3:                       {stats['assy_l4plus']:,}")
    print(f"  L4+ with blank L3, used own match:              {stats['assy_l4_own']:,}")
    print(f"  L4+ left blank (no sub group):                  {stats['assy_l4_blank']:,}")
    print(f"Saved to: {output_file}")


# ======================================================================
# PIPELINE: cost audit + highlighting  ->  sub-group mapping  ->  one final file
# ======================================================================
def run_pipeline(
    new_file="Qute EV MPL Sep26.xlsx",
    old_file="qute mpl old data.xlsx",
    audited_file="Qute_EV_MPL_Comprehensive_Audited.xlsx",   # intermediate (highlighted) file
    final_file="Qute_EV_MPL_Final.xlsx",                     # highlighted + sub-grouped
    sheet_name="1. R140B Qute EV",
    header_row=5,
):
    generate_comprehensive_bom_audit(
        new_input_file=new_file,
        old_input_file=old_file,
        output_file=audited_file,
        sheet_name=sheet_name,
        header_row=header_row,
    )
    print("\n" + "=" * 80)
    print("Feeding the highlighted file into the Sub-Group mapping ...")
    print("=" * 80)
    map_subgroups(
        new_file=audited_file,
        old_file=old_file,
        output_file=final_file,
        header_row=header_row,
    )
    print(f"\n✓ Done. Final file (highlighting + sub groups): '{final_file}'")


# ======================================================================
# INTERACTIVE INTERFACE (Jupyter / JupyterLab / JupyterLite): upload both files, click Run
# ======================================================================
def process_uploads(new_name, new_bytes, old_bytes, sheet_name, header_row, on_stage=None):
    """Save the uploaded files to a temp folder, run both parts, return (final_path, audited_path)."""
    import tempfile
    work = tempfile.mkdtemp(prefix="qute_mpl_")
    new_ext = os.path.splitext(new_name)[1].lower() or ".xlsx"
    new_path = os.path.join(work, "new_input" + new_ext)
    old_path = os.path.join(work, "old_input.xlsx")
    audited = os.path.join(work, "Qute_EV_MPL_Comprehensive_Audited.xlsx")
    final = os.path.join(work, "Qute_EV_MPL_Final.xlsx")
    with open(new_path, "wb") as fh:
        fh.write(new_bytes)
    with open(old_path, "wb") as fh:
        fh.write(old_bytes)
    stage = on_stage or (lambda *_: None)
    stage("audit")
    generate_comprehensive_bom_audit(new_input_file=new_path, old_input_file=old_path, output_file=audited,
                                     sheet_name=sheet_name, header_row=header_row)
    stage("subgroup")
    map_subgroups(new_file=audited, old_file=old_path, output_file=final, header_row=header_row)
    return final, audited


_UI_CSS = """
<style>
.qa-hero{background:radial-gradient(600px 200px at 90% -20%,rgba(90,160,255,.45),transparent 60%),linear-gradient(120deg,#071a35,#0b4a9a);
  color:#fff;border-radius:16px;padding:22px 26px;margin:4px 0 14px}
.qa-hero h2{margin:0 0 4px;font-size:22px;font-weight:800;color:#fff;letter-spacing:-.01em}
.qa-hero p{margin:0;color:#bcd0ee;font-size:13.5px}
.qa-card{border:1px solid rgba(128,128,128,.28);border-radius:14px;padding:14px 16px;margin:0 12px 12px 0;min-width:290px;flex:1}
.qa-title{font-weight:700;font-size:14.5px;margin-bottom:2px}
.qa-sub{color:#7a8599;font-size:12px;margin-bottom:10px}
.qa-chip{display:inline-block;padding:4px 11px;border-radius:999px;font-size:12px;font-weight:600;margin-left:6px}
.qa-ok{background:#d4f3df;color:#0a5c2b}.qa-none{background:rgba(128,128,128,.16);color:#7a8599}
.qa-upload button,.qa-upload .lm-Widget{border-radius:10px!important}
.qa-upload button{background:#0057b8!important;color:#fff!important;font-weight:600;border:0!important;height:36px}
.qa-run button{background:linear-gradient(120deg,#0057b8,#1a7bff)!important;color:#fff!important;border:0!important;
  border-radius:12px!important;font-weight:700;font-size:15px;height:44px;box-shadow:0 6px 16px rgba(0,87,184,.3)}
.qa-run button:disabled{opacity:.4;box-shadow:none}
.qa-legend{display:flex;flex-wrap:wrap;gap:8px 18px;font-size:12.5px;margin-top:2px}
.qa-sw{display:inline-block;width:14px;height:14px;border-radius:4px;vertical-align:-2px;margin-right:6px;border:1px solid rgba(0,0,0,.15)}
.qa-steps{display:flex;gap:8px;flex-wrap:wrap;margin:4px 0 10px}
.qa-step{flex:1;min-width:190px;border-radius:10px;padding:9px 12px;font-size:13px;font-weight:600;background:rgba(128,128,128,.14);color:#7a8599}
.qa-step.run{background:#e6f0fc;color:#0057b8}.qa-step.done{background:#d4f3df;color:#0a5c2b}.qa-step.err{background:#fde2e2;color:#a11}
.qa-stats{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px;margin:8px 0}
.qa-stat{border-radius:12px;padding:11px 14px;border:1px solid rgba(128,128,128,.25);border-left-width:6px}
.qa-stat b{display:block;font-size:24px;font-weight:800;line-height:1.15}.qa-stat span{color:#7a8599;font-size:12px}
.qa-console{background:#0d1626;border-radius:10px;padding:8px 10px;max-height:260px;overflow:auto}
.qa-console *{color:#cfe0ff!important;background:transparent!important;font-family:ui-monospace,Menlo,Consolas,monospace!important;font-size:12px!important}
.qa-dl{display:inline-block;margin:6px 10px 0 0;padding:10px 18px;border-radius:10px;background:#0057b8;color:#fff!important;
  text-decoration:none;font-weight:700;font-size:13.5px}
.qa-dl.alt{background:transparent;color:#0057b8!important;border:1.5px solid #0057b8}
.qa-err{background:#fde2e2;color:#a11;border-radius:10px;padding:10px 14px;font-weight:600;margin:6px 0}
</style>
"""


def launch_ui():
    try:
        import ipywidgets as W
        from IPython.display import display, HTML
    except ImportError:
        print("ipywidgets is required for the interface. Run  %pip install ipywidgets  in a cell, then run this again.")
        return
    import base64, shutil, traceback

    def read_upload(w):
        v = w.value
        if not v:
            return None
        if isinstance(v, dict):                      # ipywidgets 7
            item = list(v.values())[0]
            return item["metadata"]["name"], bytes(item["content"])
        item = v[0]                                  # ipywidgets 8
        return item["name"], bytes(item["content"])

    def card(title, sub, *children):
        box = W.VBox([W.HTML(f"<div class='qa-title'>{title}</div><div class='qa-sub'>{sub}</div>"), *children])
        box.add_class("qa-card")
        return box

    def chip(name=None):
        return (f"<span class='qa-chip qa-ok'>✓ {name}</span>" if name
                else "<span class='qa-chip qa-none'>no file chosen</span>")

    new_up = W.FileUpload(accept=".xlsx,.xlsb", multiple=False, description="Choose NEW file")
    old_up = W.FileUpload(accept=".xlsx", multiple=False, description="Choose OLD file")
    for u in (new_up, old_up):
        u.add_class("qa-upload"); u.layout.width = "190px"
    new_chip, old_chip = W.HTML(chip()), W.HTML(chip())

    sheet_in = W.Text(value="1. R140B Qute EV", description="Sheet", style={"description_width": "60px"},
                      layout=W.Layout(width="340px"))
    hdr_in = W.BoundedIntText(value=5, min=1, max=100, description="Header row", style={"description_width": "80px"},
                              layout=W.Layout(width="190px"))

    legend = W.HTML(
        "<div class='qa-legend'>"
        "<span><i class='qa-sw' style='background:#FFFF99'></i>Duplicate child cost under a costed parent</span>"
        "<span><i class='qa-sw' style='background:#C6EFCE'></i>Rate differs from old file</span>"
        "<span><i class='qa-sw' style='background:#BDD7EE'></i>Price change above category threshold</span></div>")

    run_btn = W.Button(description="Run audit + sub-grouping", icon="play", disabled=True,
                       layout=W.Layout(width="280px", height="44px"))
    run_btn.add_class("qa-run")

    steps = W.HTML()
    out = W.Output(); out.add_class("qa-console")
    log = W.Accordion(children=[out], selected_index=None); log.set_title(0, "Run log")
    results = W.HTML()

    def steps_html(states):
        labels = ["1 · Cost audit &amp; highlighting", "2 · Sub-group mapping", "3 · Ready to download"]
        icons = {"": "", "run": "⏳ ", "done": "✓ ", "err": "✗ "}
        return "<div class='qa-steps'>" + "".join(
            f"<div class='qa-step {s}'>{icons[s]}{l}</div>" for l, s in zip(labels, states)) + "</div>"

    steps.value = steps_html(["", "", ""])

    def refresh(_=None):
        n, o = read_upload(new_up), read_upload(old_up)
        new_chip.value, old_chip.value = chip(n[0] if n else None), chip(o[0] if o else None)
        run_btn.disabled = not (n and o)

    new_up.observe(refresh, names="value")
    old_up.observe(refresh, names="value")

    def dl(path, label, alt=False):
        with open(path, "rb") as fh:
            b64 = base64.b64encode(fh.read()).decode()
        mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
        return (f'<a class="qa-dl{" alt" if alt else ""}" download="{os.path.basename(path)}" '
                f'href="data:{mime};base64,{b64}">{label}</a>')

    def stat(n, label, color):
        return f"<div class='qa-stat' style='border-left-color:{color}'><b>{n:,}</b><span>{label}</span></div>"

    def on_run(_):
        n, o = read_upload(new_up), read_upload(old_up)
        if not (n and o):
            return
        out.clear_output(); results.value = ""; log.selected_index = None
        run_btn.disabled = True; run_btn.description = "Running ..."
        states = ["", "", ""]

        def on_stage(name):
            states[:] = ["run", "", ""] if name == "audit" else ["done", "run", ""]
            steps.value = steps_html(states)

        LAST_RUN.clear()
        with out:
            try:
                final, audited = process_uploads(n[0], n[1], o[1], sheet_in.value.strip(), int(hdr_in.value), on_stage)
                try:                                  # also keep a copy next to the notebook
                    shutil.copy(final, os.path.join(os.getcwd(), os.path.basename(final)))
                except OSError:
                    pass
                steps.value = steps_html(["done", "done", "done"])
                L = LAST_RUN
                results.value = (
                    "<div class='qa-stats'>"
                    + stat(L.get("yellow", 0), "Duplicate costs (yellow)", "#e6d600")
                    + stat(L.get("green", 0), "Rate mismatches (green)", "#4caf70")
                    + stat(L.get("blue", 0), "Threshold breaches (blue)", "#5b9bd5")
                    + stat(L.get("sub_filled", 0), f"Sub groups filled of {L.get('sub_total', 0):,} rows", "#0057b8")
                    + "</div>"
                    + dl(final, "⬇ Download final file")
                    + dl(audited, "⬇ Highlighted-only file", alt=True))
            except Exception as exc:
                states[states.index("run") if "run" in states else 0] = "err"
                steps.value = steps_html(states)
                results.value = f"<div class='qa-err'>The run failed: {type(exc).__name__}: {exc}</div>"
                log.selected_index = 0
                traceback.print_exc()
        run_btn.description = "Run audit + sub-grouping"
        refresh()

    run_btn.on_click(on_run)

    display(HTML(_UI_CSS))
    display(W.VBox([
        W.HTML("<div class='qa-hero'><h2>Qute EV MPL · Audit &amp; Sub-grouping</h2>"
               "<p>Upload the new and old MPL files, then run. You get one workbook with cost highlighting and sub groups.</p></div>"),
        W.HBox([
            card("1 · New MPL file", "The file to audit (.xlsx or .xlsb)", W.HBox([new_up, new_chip])),
            card("2 · Old MPL file", "Reference data (.xlsx)", W.HBox([old_up, old_chip])),
        ], layout=W.Layout(flex_flow="row wrap")),
        card("3 · Settings", "Change only if your sheet differs", W.HBox([sheet_in, hdr_in], layout=W.Layout(flex_flow="row wrap"))),
        card("Colour key", "How rows are highlighted in the output", legend),
        run_btn, steps, results, log,
    ]))


# --- Execution ---
if __name__ == "__main__":
    launch_ui()